In [ ]:
import warnings
from typing import Tuple,List
from copy import deepcopy
import numpy as np

class Parameter:
    """
    Classe che rappresenta un singolo parametro di un modello.

    Attributes:
        name (str): Nome del parametro.
        value (float): Valore del parametro.
        frozen (bool): Stato di congelamento del parametro.
        bounds (Tuple[float, float]): Limiti del parametro.
        description (str): Descrizione del parametro.
    """

    def __init__(
        self,
        name: str,
        value,
        frozen: bool = False,
        bounds = (-float('inf'), float('inf')),
        description: str = "",
    ) -> None:
        """
        Inizializza un nuovo parametro.

        Args:
            name (str): Nome del parametro.
            value (float): Valore del parametro.
            frozen (bool, opzionale): Stato di congelamento del parametro. Default è False.
            bounds (Tuple[float, float], opzionale): Limiti del parametro. Default è (-inf, inf).
            description (str, opzionale): Descrizione del parametro. Default è "".

        Raises:
            TypeError: Se i tipi degli argomenti non sono corretti.
            ValueError: Se i valori degli argomenti non sono validi.
        """
        ParameterValidator.validate_name(name)
        ParameterValidator.validate_bounds(bounds)
        ParameterValidator.validate_value_in_bounds(value, bounds)

        self._name = name
        self._value = value
        self._frozen = frozen
        self._bounds = bounds
        self._description = description

    @property
    def name(self) -> str:
        """
        Ritorna il nome del parametro.

        Returns:
            str: Nome del parametro.
        """
        return self._name

    @name.setter
    def name(self, new_name: str) -> None:
        """
        Imposta un nuovo nome per il parametro.

        Args:
            new_name (str): Nuovo nome del parametro.

        Raises:
            TypeError: Se il nuovo nome non è una stringa.
        """
        ParameterValidator.validate_name(new_name)
        self._name = new_name

    @property
    def value(self) -> float:
        """
        Ritorna il valore del parametro.

        Returns:
            float: Valore del parametro.
        """
        return self._value

    @value.setter
    def value(self, new_value: float) -> None:
        """
        Imposta un nuovo valore per il parametro.

        Args:
            new_value (float): Nuovo valore del parametro.

        Raises:
            ValueError: Se il parametro è congelato o il nuovo valore è fuori dai limiti.
        """
        if self.frozen is True:
            warnings.warn(
                f"Parameter {self.name} is frozen, new value will be ignored!"
            )
            return
        ParameterValidator.validate_value_in_bounds(new_value, self._bounds)
        self._value = new_value

    @property
    def bounds(self) -> Tuple[float, float]:
        """
        Ritorna i limiti del parametro.

        Returns:
            Tuple[float, float]: Limiti del parametro.
        """
        return self._bounds

    @bounds.setter
    def bounds(self, new_bounds: Tuple[float, float]) -> None:
        """
        Imposta nuovi limiti per il parametro.

        Args:
            new_bounds (Tuple[float, float]): Nuovi limiti del parametro.

        Raises:
            TypeError: Se i nuovi limiti non sono una tupla di due elementi.
            ValueError: Se i nuovi limiti non sono validi.
        """
        if self.frozen is True:
            warnings.warn(
                f"Parameter {self.name} is frozen, new bounds will be ignored!"
            )
            return
        ParameterValidator.validate_bounds(new_bounds)
        ParameterValidator.validate_value_in_bounds(self._value, new_bounds)
        self._bounds = new_bounds

    @property
    def frozen(self) -> bool:
        """
        Ritorna lo stato di congelamento del parametro.

        Returns:
            bool: True se il parametro è congelato, False altrimenti.
        """
        return self._frozen

    @frozen.setter
    def frozen(self, is_true: bool) -> None:
        """
        Imposta lo stato di congelamento del parametro.

        Args:
            is_true (bool): Stato di congelamento del parametro.

        Raises:
            TypeError: Se il valore non è un booleano.
        """
        ParameterValidator.validate_frozen(is_true)
        self._frozen = is_true

    @property
    def description(self) -> str:
        """
        Ritorna la descrizione del parametro.

        Returns:
            str: Descrizione del parametro.
        """
        return self._description

    @description.setter
    def description(self, str: str) -> None:
        """
        Imposta una nuova descrizione per il parametro.

        Args:
            str (str): Nuova descrizione del parametro.

        Raises:
            TypeError: Se la descrizione non è una stringa.
        """
        ParameterValidator.validate_description(str)
        self._description = str

    def copy(self) -> "Parameter":
        """
        Ritorna una copia del parametro.

        Returns:
            Parameter: Copia del parametro.
        """
        return deepcopy(self)

    def __len__(self) -> int:
        """
        Ritorna la lunghezza del parametro (sempre 1).

        Returns:
            int: Lunghezza del parametro.
        """
        return 1

    def __iter__(self):
        """
        Ritorna un iteratore per il parametro.
        inutile(?)

        Returns:
            Iterator: Iteratore per il parametro.
        """
        return iter(
            [self._name, self._value, self._frozen, self._bounds, self._description]
        )

    def __getitem__(self, key):
        """
        Ritorna l'attributo specificato del parametro.

        Args:
            key (str): Nome dell'attributo.

        Returns:
            Any: Valore dell'attributo.
        """
        return getattr(self, key)

    def __setitem__(self, key, value) -> None:
        """
        Imposta l'attributo specificato del parametro.

        Args:
            key (str): Nome dell'attributo.
            value (Any): Valore da assegnare all'attributo.
        """
        setattr(self, key, value)

    def __str__(self) -> str:
        """
        Ritorna una rappresentazione testuale del parametro.

        Returns:
            str: Rappresentazione testuale del parametro.
        """
        total_string = f"PARAM NAME: {self.name}\n"
        total_string += "-" * 60 + "\n"
        total_string += f"{'NOME':<15} {'VALORE':<10} {'FREEZE':<10} {'BOUNDS':<20} {'DESCRIZIONE':<20} \n"
        total_string += "-" * 60 + "\n"

        value_str = f"{self._value:.5g}"
        bounds_str = f"({self._bounds[0]:.5g}, {self._bounds[1]:.5g})"
        total_string += f"{self.name:<15} {value_str:<10} {self._frozen:<10} {bounds_str:<20} {self.description:<20} \n"
        return total_string


class ParameterValidator:
    """
    Classe per la gestione della validazione di parametri singoli.
    """

    @staticmethod
    def validate_name(name: str) -> None:
        """
        Valida il nome del parametro.

        Args:
            name (str): Nome del parametro.

        Raises:
            TypeError: Se il nome non è una stringa.
        """
        if not isinstance(name, str):
            raise TypeError("Il nome del parametro deve essere una stringa!")

    @staticmethod
    def validate_bounds(bounds: Tuple[float, float]) -> None:
        """
        Valida i limiti del parametro.

        Args:
            bounds (Tuple[float, float]): Limiti del parametro.

        Raises:
            TypeError: Se i limiti non sono una tupla di due elementi.
            ValueError: Se i limiti non sono validi.
        """
        if not isinstance(bounds, (list, np.ndarray, tuple)):
            raise TypeError(
                f"New bounds must be in form of iterable, you gave {type(bounds)}"
            )
        if len(bounds) != 2:
            raise ValueError("I limiti devono essere una tupla con due elementi.")
        if not bounds[0] <= bounds[1]:
            raise ValueError(
                "Il limite inferiore deve essere minore o uguale al limite superiore."
            )

    @staticmethod
    def validate_value_in_bounds(value: float, bounds: Tuple[float, float]) -> None:
        """
        Valida che il valore del parametro sia entro i limiti specificati.

        Args:
            value (float): Valore del parametro.
            bounds (Tuple[float, float]): Limiti del parametro.

        Raises:
            TypeError: Se il valore non è un numero.
            ValueError: Se il valore è fuori dai limiti.
        """
        if not isinstance(value, (int, float)):
            raise TypeError("Value must be of type Number, not ", type(value))
        if not bounds[0] <= value <= bounds[1]:
            raise ValueError(f"Il valore {value} è fuori dai limiti {bounds}")

    @staticmethod
    def validate_frozen(is_true: bool) -> None:
        """
        Valida lo stato di congelamento del parametro.

        Args:
            is_true (bool): Stato di congelamento del parametro.

        Raises:
            TypeError: Se il valore non è un booleano.
        """
        if not isinstance(is_true, bool):
            raise TypeError(
                f'Il valore di "frozen" può essere solo True o False, hai fornito {is_true, type(is_true)}'
            )

    @staticmethod
    def validate_description(strg: str) -> None:
        """
        Valida la descrizione del parametro.

        Args:
            strg (str): Descrizione del parametro.

        Raises:
            TypeError: Se la descrizione non è una stringa.
        """
        if not isinstance(strg, str):
            raise TypeError("Description must be a string!")
